# Steam Review Analysis

## 1. Introduction & Objective

### 1.1. Background

This project explores a sample of the Steam Reviews Dataset containing information about reviewers, their game playtime, review activity, review feedback, and the text of their written reviews.<br>
<br>
The dataset contains both structured variables, such as playtime and the number of games owned, and unstructured textual data in the form of user reviews. This makes it possible to investigate both user behavior and review content.<br>
<br>
You can find out more information about the Steam Reviews Dataset here:
https://www.kaggle.com/datasets/forgemaster/steam-reviews-dataset?resource=download

### 1.2. Objective

The main objective of this project is to investigate which characteristics of Steam reviews and reviewers are associated with positive game recommendations. In addition, the project explores whether review text provides useful information for predicting whether a user recommends a game.<br>
<br>
Because this analysis uses a sample of the full Steam Reviews Dataset, the findings should be interpreted as observations about the analyzed sample rather than conclusions about all Steam users or reviews.

### 1.3. Research Questions

**Primary Research Question**<br>
<br>
1. What factors are associated with whether a Steam user recommends a game?<br>
<br>
**Secondary Research Questions**<br>
<br>
2. How is a reviewer's playtime at the time of writing the review related to the likelihood of recommending the game? <br>
3. What characteristics of a review are associated with receiving more helpful votes?<br>
4. Does incorporating review text improve the prediction of whether a user recommends a game compared with using structured data alone?<br>

### 1.4. Analytical Approach

To investigate these questions, the project will proceed through data quality assessment, exploratory data analysis, feature engineering, text analysis, predictive modeling, and model evaluation. The results will then be interpreted in the context of the research questions, with particular attention to the limitations of the sample and the available variables.

## 2. Loading the Data

First we import the required libraries:

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import seaborn as sns

We'll now load the dataset:

In [2]:
df = pd.read_csv("../data/reviews-13500-13537.csv")

Let's make sure it got loaded correctly:

In [3]:
df.shape

(8086, 13)

Let's also take a look at the first few rows of the dataset and view its statistic overview:

In [4]:
df.head(5)

,steamid,appid,voted_up,votes_up,votes_funny,weighted_vote_score,playtime_forever,playtime_at_review,num_games_owned,num_reviews,review,unix_timestamp_created,unix_timestamp_updated
0,76561198414702117,598060,False,0,1,0.424996,368,192,188,14,Game is amazing. But there is a camera sway wh...,1618673751,1618673751
1,76561198422616434,598060,True,0,0,0.000000,1175,1133,39,5,YES.,1618588463,1618588463
2,76561198001462345,598060,True,0,0,0.000000,63,63,324,2,Very Cool concept but with the platforming and...,1618373962,1618373962
3,76561198008037092,598060,True,0,0,0.000000,473,452,431,16,"Yup, pretty hard. I like it. I hope I play it ...",1618337175,1618337175
4,76561198105898959,598060,True,0,0,0.000000,3139,2699,67,1,Wonderful game to just turn off your mind and ...,1618193343,1618193343


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8086 entries, 0 to 8085
Data columns (total 13 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   steamid                 8086 non-null   int64  
 1   appid                   8086 non-null   int64  
 2   voted_up                8086 non-null   bool   
 3   votes_up                8086 non-null   int64  
 4   votes_funny             8086 non-null   int64  
 5   weighted_vote_score     8086 non-null   float64
 6   playtime_forever        8086 non-null   int64  
 7   playtime_at_review      8086 non-null   int64  
 8   num_games_owned         8086 non-null   int64  
 9   num_reviews             8086 non-null   int64  
 10  review                  8070 non-null   str    
 11  unix_timestamp_created  8086 non-null   int64  
 12  unix_timestamp_updated  8086 non-null   int64  
dtypes: bool(1), float64(1), int64(10), str(1)
memory usage: 766.1 KB


## 3. Data Understanding & Data Quality

### 3.1. Understanding the Variables

| **Column** | **Meaning** | **Type/Role**
| :--- | :--- | :--- |
| `steamid`| The Steam ID of the user who wrote this review | Identifier |
| `appid` | The ID of the app being reviewed | Identifier |
| `voted_up` | Whether the review showed a positive attitude towards the game. **True** means the review was positive | Target Variable |
| `votes_up` | The number of other users who found this review helpful | Numerical |
| `votes_funny` | The number of other users who found this review funny | Numerical |
| `weighted_vote_score` | Helpfulness score, automatically calculated by Steam using its internal algorithm | Numerical |
| `playtime_forever` | How many hours the user who wrote this review had played this game when the dataset was created | Numerical |
| `playtime_at_review` | How many hours the user had played this game when writing the review | Numerical |
| `num_games_owned` | The number of games the user owns | Numerical |
| `num_reviews` | The number of reviews the user has written | Numerical |
| `review` | The text of the written review | **Text** |
| `unix_timestamp_created` | The date on which this review was created | Timestamp |
| `unix_timestamp_updated` | The date on which this review was last updated | Timestamp |

### 3.2. Data types and basic structure

In [6]:
df.shape

(8086, 13)

In [7]:
df.dtypes

steamid                     int64
appid                       int64
voted_up                     bool
votes_up                    int64
votes_funny                 int64
weighted_vote_score       float64
playtime_forever            int64
playtime_at_review          int64
num_games_owned             int64
num_reviews                 int64
review                        str
unix_timestamp_created      int64
unix_timestamp_updated      int64
dtype: object

The dataset contains 8086 rows and 13 columns. 11 columns are numerical, including 10 integer columns and 1 float column. 1 column, the `voted_up` column, is boolean. And 1 column, the `review` column, is text.

In [8]:
df.nunique()

steamid                   8042
appid                       12
voted_up                     2
votes_up                    89
votes_funny                 34
weighted_vote_score       2349
playtime_forever          3623
playtime_at_review        2608
num_games_owned           1185
num_reviews                282
review                    7823
unix_timestamp_created    8086
unix_timestamp_updated    8086
dtype: int64

In [9]:
df.describe()

,steamid,appid,votes_up,votes_funny,weighted_vote_score,playtime_forever,playtime_at_review,num_games_owned,num_reviews,unix_timestamp_created,unix_timestamp_updated
count,8.086000e+03,8086.000000,8086.000000,8086.000000,8086.000000,8086.000000,8086.000000,8086.000000,8086.000000,8.086000e+03,8.086000e+03
mean,7.656120e+16,598622.122186,2.424314,0.376700,0.223451,2795.494558,1393.070121,274.526960,25.050952,1.567183e+09,1.568417e+09
std,2.765839e+08,338.215292,12.078875,2.325907,0.259796,8303.102209,4352.532906,487.213087,89.577746,4.081779e+07,4.040761e+07
min,7.656120e+16,598060.000000,0.000000,0.000000,0.000000,5.000000,5.000000,0.000000,1.000000,1.487905e+09,1.487905e+09
25%,7.656120e+16,598330.000000,0.000000,0.000000,0.000000,350.000000,190.000000,54.000000,3.000000,1.531861e+09,1.533660e+09
50%,7.656120e+16,598700.000000,0.000000,0.000000,0.000000,869.000000,452.500000,138.000000,7.000000,1.580057e+09,1.581076e+09
75%,7.656120e+16,598980.000000,1.000000,0.000000,0.497512,2244.000000,1111.000000,309.000000,20.000000,1.605924e+09,1.606098e+09
max,7.656120e+16,598980.000000,545.000000,79.000000,0.941871,239974.000000,100120.000000,12696.000000,4137.000000,1.619058e+09,1.619058e+09


We have 8086 reviews but 8042 users, indicating that several users wrote multiple reviews in this sample. We can view the userid values of some users who wrote the most reviews in this sample:

In [10]:
df["steamid"].value_counts().head()

steamid
76561197965193437    3
76561197980787602    2
76561198045592963    2
76561198001558786    2
76561197993894575    2
Name: count, dtype: int64

The `appid` column only has 12 distinct values, while the dataset as a whole reportedly contains reviews of 8183 games. Since our sample is much smaller than the original dataset, we should be careful about making statements about "Steam games" in general just from observations taken from this sample.<br>
We can view the unique values in the column `appid` and their counts in our sample:

In [11]:
df["appid"].value_counts()

appid
598980    2341
598330    1892
598060    1097
598700    1055
598960     660
598810     528
598450     161
598470     115
598490      87
598690      70
598440      63
598070      17
Name: count, dtype: int64

The column `voted_up` has 2 distinct values, which is exactly what we would expect from a boolean column with only 2 possible values: True and False. However, we can't yet tell whether these values are balanced. Let's check it:

In [12]:
df["voted_up"].value_counts(normalize=True) * 100

voted_up
True     86.730151
False    13.269849
Name: proportion, dtype: float64

Approximately 86.73% of the observations in this sample has `voted_up`=True, which means our sample is highly skewed.

`votes_up` only has 89 distinct values even though there are 8086 rows in this dataset. The min value and the 25%  and 50% quartiles for this column are all 0, indicating that at least 50% of the reviews in this sample has no up vote. We can verify that here:

In [13]:
df["votes_up"].value_counts(normalize=True) * 100

votes_up
0      58.483799
1      17.214939
2       7.766510
3       4.044027
4       2.188969
         ...    
290     0.012367
75      0.012367
71      0.012367
73      0.012367
97      0.012367
Name: proportion, Length: 89, dtype: float64

Similarly, `votes_funny` only has 34 distinct values, and the min value and all 3 quartiles are 0, which means at least 75% of the reviews in our sample had no funny vote. The actual percentage turns out to be around 86.58%:

In [14]:
df["votes_funny"].value_counts(normalize=True).head() * 100

votes_funny
0    86.581746
1     8.372496
2     1.941628
3     0.939896
4     0.581252
Name: proportion, dtype: float64

`weighted_vote_score` has 2349 distinct values ranging from 0 to 0.941871. This variable has much more variation and can potentially be useful for analyzing review helpfulness.

`playtime_forever` and `playtime_at_review` also has 3623 and 2608 distinct values respectively. These two variables have great variation, with `playtime_at_review` potentially being more relevant to the research question.

`num_games_owned` has 1185 distinct values, while `num_reviews` only has 282 distinct values, with a maximum of 4,137 reviews written by a single user. This might be worth further investigation.

We have 7823 unique review texts but 8086 data rows, meaning there are 263 duplicated values in the `review` column. However, this doesn't necessarily mean there were duplicate observations, as two users can independently write a review like 'Good game' or 'YES'. We can look at that later.

Both timestamp columns has 8086 unique values corresponding to our 8086 data rows, which means all the reviews were written and last updated at different times compared to each other.

Let's take another look at the statistical summary of the numerical columns in the dataset:

In [15]:
df.describe()

,steamid,appid,votes_up,votes_funny,weighted_vote_score,playtime_forever,playtime_at_review,num_games_owned,num_reviews,unix_timestamp_created,unix_timestamp_updated
count,8.086000e+03,8086.000000,8086.000000,8086.000000,8086.000000,8086.000000,8086.000000,8086.000000,8086.000000,8.086000e+03,8.086000e+03
mean,7.656120e+16,598622.122186,2.424314,0.376700,0.223451,2795.494558,1393.070121,274.526960,25.050952,1.567183e+09,1.568417e+09
std,2.765839e+08,338.215292,12.078875,2.325907,0.259796,8303.102209,4352.532906,487.213087,89.577746,4.081779e+07,4.040761e+07
min,7.656120e+16,598060.000000,0.000000,0.000000,0.000000,5.000000,5.000000,0.000000,1.000000,1.487905e+09,1.487905e+09
25%,7.656120e+16,598330.000000,0.000000,0.000000,0.000000,350.000000,190.000000,54.000000,3.000000,1.531861e+09,1.533660e+09
50%,7.656120e+16,598700.000000,0.000000,0.000000,0.000000,869.000000,452.500000,138.000000,7.000000,1.580057e+09,1.581076e+09
75%,7.656120e+16,598980.000000,1.000000,0.000000,0.497512,2244.000000,1111.000000,309.000000,20.000000,1.605924e+09,1.606098e+09
max,7.656120e+16,598980.000000,545.000000,79.000000,0.941871,239974.000000,100120.000000,12696.000000,4137.000000,1.619058e+09,1.619058e+09


`playtime_forever` and `playtime_at_review` both have the mean more than 3 times higher than the median, suggesting that some users probably have much more playtime than others and are raising the average playtime.

`num_reviews` has a median of 7 but a much higher max value of 4137, possibly making the distribution of this variable highly right-skewed.

`votes_up` has a median of 0, a 75% quartile of 1, a mean of 2.42, and a max value of 545. This means that 50% of reviews have votes_up = 0, 75% of reviews have votes_up <= 1, but at least 1 review has votes_up = 545. This variable has a **right-skewed distribution.**
The same pattern can be seen in the `votes_funny` column, with a median of 0, a 75% quartile of 0, and a max value of 79.

`weighted_vote_score` has a mean value of 0, a median of 0, and a 75% quartile of 0.497512. 50% of the reviews has weighted_vote_score = 0, but 75% of the record has weighted_vote_score <= 0.497512, suggesting that the distribution of this variable might be right-skewed as well.

The mean, median, min and max values for `appid` are rather close and not particularly meaningful since this is an identifier, not a meaningful variable. The same goes for `steamid`.

`unix_timestamp_created` and `unix_timestamp_updated` are still numerical, which is why they appear in this statistical summary. We will need to convert them to DateTime format before investigating them.

Overall, we have learned that:
- Our sample has 8086 reviews and 8042 unique users but only **12 games.** This number of games is much smaller than the reported count for the original dataset as a whole, so we have to be cautious when making statements about Steam games in general.
- We have 7823 unique review texts.
- We do not have any obvious missing values in the numerical columns.
- Several numerical variables are strongly **right-skewed,** such as `num_reviews`, `votes_up` or `votes_funny`.
- `votes_up`, `votes_funny`, and `weighted_vote_score` contains many zero values. `playtime_forever`, `playtime_at_review`, and `num_review` have extreme values.
- Timestamps need conversion before they can be interpreted.
- `steamid` and `appid` are identifiers, not meaningful numerical measurements.
- Our target variable `voted_up` is also greatly skewed.

### 3.3. Data Quality Assessment

#### Missing values:

In [16]:
df.isna().sum()

steamid                    0
appid                      0
voted_up                   0
votes_up                   0
votes_funny                0
weighted_vote_score        0
playtime_forever           0
playtime_at_review         0
num_games_owned            0
num_reviews                0
review                    16
unix_timestamp_created     0
unix_timestamp_updated     0
dtype: int64

We only have 16 missing valies in the `review` column.

#### Duplicate Observations:

In [17]:
df.duplicated().sum()

np.int64(0)

We don't have any duplicated records, which is a good sign.

#### Invalid/unusual values:

##### Boolean column: `voted_up`

In [20]:
df["voted_up"].value_counts()

voted_up
True     7013
False    1073
Name: count, dtype: int64

We have no unusual or invalid values in the `voted_up` column since it consists entirely of True and False values.

##### Numerical variables

Let's check for negative values:

In [22]:
numeric_nonnegative = [
    "votes_up",
    "votes_funny",
    "playtime_forever",
    "playtime_at_review",
    "num_games_owned",
    "num_reviews"
]

for col in numeric_nonnegative:
    print(col, (df[col] < 0).sum())

votes_up 0
votes_funny 0
playtime_forever 0
playtime_at_review 0
num_games_owned 0
num_reviews 0


For the numerical columns that aren't supposed to be negative, we don't have any negative values.

Let's also check whether `playtime_at_review` exceeds `playtime_forever`, since the former should conceptually not surpass the latter:

In [23]:
(df["playtime_at_review"] > df["playtime_forever"]).sum()

np.int64(1)

Oh no! Let's check this anomalous record:

In [28]:
df.loc[
    df["playtime_at_review"] > df["playtime_forever"],
    [
        "steamid",
        "appid",
        "voted_up",
        "playtime_forever",
        "playtime_at_review",
        "review",
        "unix_timestamp_created",
        "unix_timestamp_updated"
    ]
]

,steamid,appid,voted_up,playtime_forever,playtime_at_review,review,unix_timestamp_created,unix_timestamp_updated
4653,76561198127270528,598810,True,150,183,very addictive,1609159372,1609159372


**Logical consistency:** One observation was identified where playtime_at_review (183 hours) exceeds playtime_forever (150 hours). This contradicts the expected relationship between the two variables. However, because this is a single observation and there is insufficient evidence to determine which value is incorrect, the observation is retained rather than arbitrarily modified or removed. It will be treated as a potential anomaly when analyzing playtime-related variables.

#### Target-variable distribution:

In [21]:
df['voted_up'].value_counts()

voted_up
True     7013
False    1073
Name: count, dtype: int64

As we've discussed above, our target variable `voted_up` in this sample is **highly skewed,** with approximately 86.73% of the records having voted_up = True.<br>
However, this bias is not necessarily a sampling problem. It might actually represent the overall trend, since it's not entirely impossible for a majority of Steam reviews to be positive.

#### Numerical-variable distribution:

In [24]:
df.describe()

,steamid,appid,votes_up,votes_funny,weighted_vote_score,playtime_forever,playtime_at_review,num_games_owned,num_reviews,unix_timestamp_created,unix_timestamp_updated
count,8.086000e+03,8086.000000,8086.000000,8086.000000,8086.000000,8086.000000,8086.000000,8086.000000,8086.000000,8.086000e+03,8.086000e+03
mean,7.656120e+16,598622.122186,2.424314,0.376700,0.223451,2795.494558,1393.070121,274.526960,25.050952,1.567183e+09,1.568417e+09
std,2.765839e+08,338.215292,12.078875,2.325907,0.259796,8303.102209,4352.532906,487.213087,89.577746,4.081779e+07,4.040761e+07
min,7.656120e+16,598060.000000,0.000000,0.000000,0.000000,5.000000,5.000000,0.000000,1.000000,1.487905e+09,1.487905e+09
25%,7.656120e+16,598330.000000,0.000000,0.000000,0.000000,350.000000,190.000000,54.000000,3.000000,1.531861e+09,1.533660e+09
50%,7.656120e+16,598700.000000,0.000000,0.000000,0.000000,869.000000,452.500000,138.000000,7.000000,1.580057e+09,1.581076e+09
75%,7.656120e+16,598980.000000,1.000000,0.000000,0.497512,2244.000000,1111.000000,309.000000,20.000000,1.605924e+09,1.606098e+09
max,7.656120e+16,598980.000000,545.000000,79.000000,0.941871,239974.000000,100120.000000,12696.000000,4137.000000,1.619058e+09,1.619058e+09


As we mentioned above, several numerical variables exhibit strong right-skewed distributions. For example, `playtime_forever` has a median of 869 minutes compared with a mean of 2,795 minutes, while `num_reviews` has a median of 7 compared with a mean of 25.1. This suggests that a relatively small number of highly active users or reviews may strongly influence the mean. These distributions will be examined visually during exploratory data analysis.

#### Timestamp validity:

First, let's check whether the timestamps are valid:

In [25]:
df["created_at"] = pd.to_datetime(
    df["unix_timestamp_created"],
    unit="s"
)

df["updated_at"] = pd.to_datetime(
    df["unix_timestamp_updated"],
    unit="s"
)

df[["created_at", "updated_at"]].describe()

,created_at,updated_at
count,8086,8086
mean,2019-08-30 16:32:57,2019-09-13 23:21:39
min,2017-02-24 03:00:18,2017-02-24 03:00:18
25%,2018-07-17 21:01:28,2018-08-07 16:41:18
50%,2020-01-26 16:42:54,2020-02-07 11:41:32
75%,2020-11-21 01:54:28,2020-11-23 02:23:30
max,2021-04-22 02:11:55,2021-04-22 02:11:55


In [26]:
print(df["created_at"].min(), df["created_at"].max())
print(df["updated_at"].min(), df["updated_at"].max())

2017-02-24 03:00:18 2021-04-22 02:11:55
2017-02-24 03:00:18 2021-04-22 02:11:55


In [27]:
(df["updated_at"] < df["created_at"]).sum()

np.int64(0)

The Unix timestamps were converted into datetime values for easier interpretation. The resulting dates fall within a plausible historical range, and no observations have an update timestamp earlier than their creation timestamp. Therefore, no obvious timestamp inconsistencies were detected.

#### Sample/representativeness considerations:

This analysis uses one file from the larger dataset. The sample contains 8,086 reviews across 12 games and hence should not automatically be assumed to represent the overall Steam review population. The results should therefore be interpreted as observations about this sample rather than definitive conclusions about all Steam users or games.

## 4. Exploratory Data Analysis

## 5. Feature Engineering

## 6. Review Text Analysis / NLP

## 7. Modeling

## 8. Model Evaluation

## 9. Interpretation & Findings

## 10. Limitations & Future Work

## 11. Conclusion